# Saneamento e Correção de Labels

In [1]:
import pandas as pd

file_name = 'FRUTAS_DB.novo_monitoramento.csv'
df = pd.read_csv(file_name)

df['estado_real'] = df['estado_real'].astype(str).str.lower().str.strip()

mapeamento = {
    'alerta': 'Alerta',
    'madura': 'Madura',
    'risco de perda': 'Risco de Perda',
    'sem risco': 'Sem Risco'
}
df['estado_real'] = df['estado_real'].map(mapeamento)

clean_file = 'FRUTAS_DB_limpo.csv'
df.to_csv(clean_file, index=False)

print(f" Saneamento concluído! Arquivo '{clean_file}' gerado.")
print("Classes após limpeza:", df['estado_real'].unique())

 Saneamento concluído! Arquivo 'FRUTAS_DB_limpo.csv' gerado.
Classes após limpeza: <ArrowStringArray>
['Sem Risco', 'Alerta', 'Risco de Perda', 'Madura']
Length: 4, dtype: str


# Validação Estrutural e Conversão Numérica

In [ ]:
import pandas as pd

df = pd.read_csv('FRUTAS_DB_limpo.csv')

print("=== VALIDAÇÃO ESTRUTURAL ===\n")

colunas_necessarias = [
    '_id',
    'tipoFruta',
    'temperatura',
    'umidade_ar',
    'mq3_raw',
    'lote',
    'estado_real'
]

faltando = [c for c in colunas_necessarias if c not in df.columns]

if faltando:
    print(f"Colunas ausentes: {faltando}")
else:
    print("Todas as colunas obrigatórias estão presentes.")


colunas_numericas = [
    'temperatura',
    'umidade_ar',
    'mq3_raw'
]

for col in colunas_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("\nConversão numérica concluída.")

print("\n=== NaN CRÍTICOS ===")

nan_criticos = df[
    colunas_numericas + ['estado_real']
].isnull().sum()

print(nan_criticos)

labels_validas = [
    'Alerta',
    'Madura',
    'Risco de Perda',
    'Sem Risco'
]

labels_invalidas = df[
    ~df['estado_real'].isin(labels_validas)
]['estado_real'].unique()

print("\n=== LABELS INVÁLIDAS ===")
print(labels_invalidas)


print("\n=== POSSÍVEIS OUTLIERS FÍSICOS ===")

outliers = df[
    (df['temperatura'] < 0) |
    (df['temperatura'] > 60) |
    (df['umidade_ar'] < 0) |
    (df['umidade_ar'] > 100) |
    (df['mq3_raw'] < 0)
]

print(f"Quantidade encontrada: {len(outliers)}")


df['estado_real_original'] = df['estado_real']

print("\nBackup científico criado: estado_real_original")


validated_file = 'FRUTAS_DB_validado.csv'
df.to_csv(validated_file, index=False)

print(f"\n Arquivo validado salvo: {validated_file}")

# Correção Inteligente de Nomeações por Monitoramento

In [3]:
import pandas as pd


# Carregar base validada
df = pd.read_csv('FRUTAS_DB_validado.csv')

print("=== INICIANDO AUDITORIA CLIMATÉRICA ===\n")


id_inicio = '69c2ea664f3e7eb1d6eb0fd4'
id_fim = '69c7fa9b4f3e7eb1d6eb1426'

idx_inicio = df.index[df['_id'] == id_inicio].tolist()
idx_fim = df.index[df['_id'] == id_fim].tolist()

if idx_inicio and idx_fim:

    start = idx_inicio[0]
    end = idx_fim[0]

    # Corrige apenas o intervalo afetado
    df.loc[start:end, 'tipoFruta'] = 'banana'

    print(" Correção de tipoFruta concluída")
    print(f"   Intervalo corrigido: linhas {start} até {end}")
    print(f"   Total afetado: {(end - start) + 1} linhas\n")

else:
    print(" IDs de referência não encontrados.\n")


correcoes_realizadas = 0
linhas_corrigidas = []


for idx, row in df.iterrows():


    if row['lote'] not in ['lote_L', 'lote_M']:
        continue


    if pd.isna(row['mq3_raw']) or pd.isna(row['temperatura']) or pd.isna(row['umidade_ar']):
        continue

    mq3 = row['mq3_raw']
    temp = row['temperatura']
    umi = row['umidade_ar']

    estado_atual = row['estado_real']

    
    # BAIXA EMISSÃO DE GÁS
    
    if mq3 < 1500:
        estado_sugerido = 'Alerta' if temp > 30 else 'Sem Risco'
        estados_aceitaveis = [
            'Sem Risco',
            'Alerta'
        ]

    
    # PICO CLIMATÉRICO
    
    elif 1500 <= mq3 < 2800:
        estado_sugerido = 'Alerta' if temp > 28 else 'Madura'
        estados_aceitaveis = [
            'Madura',
            'Alerta',
            'Sem Risco'
        ]
    
    # FERMENTAÇÃO / DETERIORAÇÃO
    
    else:
        estado_sugerido = 'Risco de Perda'
        estados_aceitaveis = [
            'Risco de Perda',
            'Madura',
            'Alerta'
        ]
        if estado_atual == 'Sem Risco':
            estados_aceitaveis = []
    if estado_atual not in estados_aceitaveis:
        valor_antigo = estado_atual
        df.at[idx, 'estado_real'] = estado_sugerido
        correcoes_realizadas += 1
        linhas_corrigidas.append({
            'linha': idx,
            'lote': row['lote'],
            'mq3_raw': mq3,
            'temperatura': temp,
            'umidade_ar': umi,
            'antes': valor_antigo,
            'depois': estado_sugerido
        })


print("=== RESULTADO DA AUDITORIA ===\n")

print(f" Correções seletivas realizadas: {correcoes_realizadas}\n")

# Mostrar exemplos das correções
if len(linhas_corrigidas) > 0:
    print("=== EXEMPLOS DE LINHAS CORRIGIDAS ===\n")
    preview = pd.DataFrame(linhas_corrigidas)
    display(preview.head(10))

else:
    print(" Nenhuma inconsistência física grave encontrada.\n")


clean_file = 'FRUTAS_DB_limpo-corrigido.csv'
df.to_csv(clean_file, index=False)

print("=== EXPORTAÇÃO FINAL ===\n")
print(f" Arquivo salvo: {clean_file}")
print("\n=== DISTRIBUIÇÃO FINAL DOS ESTADOS ===\n")
print(df['estado_real'].value_counts())
print("\n=== PROCESSO CONCLUÍDO COM SUCESSO ===")

=== INICIANDO AUDITORIA CLIMATÉRICA ===

✅ Correção de tipoFruta concluída
   Intervalo corrigido: linhas 958 até 2064
   Total afetado: 1107 linhas

=== RESULTADO DA AUDITORIA ===

✅ Correções seletivas realizadas: 592

=== EXEMPLOS DE LINHAS CORRIGIDAS ===



,linha,lote,mq3_raw,temperatura,umidade_ar,antes,depois
0,958,lote_L,377,28.5,77,Madura,Sem Risco
1,959,lote_L,407,28.5,79,Madura,Sem Risco
2,960,lote_L,423,28.1,84,Madura,Sem Risco
3,961,lote_L,431,28.0,88,Madura,Sem Risco
4,962,lote_L,439,28.0,89,Madura,Sem Risco
5,963,lote_L,450,28.0,91,Madura,Sem Risco
6,964,lote_L,451,28.2,91,Madura,Sem Risco
7,965,lote_L,454,28.5,91,Madura,Sem Risco
8,966,lote_L,454,28.5,92,Madura,Sem Risco
9,967,lote_L,432,28.5,92,Madura,Sem Risco


=== EXPORTAÇÃO FINAL ===

✅ Arquivo salvo: FRUTAS_DB_limpo-corrigido.csv

=== DISTRIBUIÇÃO FINAL DOS ESTADOS ===

estado_real
Madura            1883
Risco de Perda    1608
Sem Risco         1171
Alerta             593
Name: count, dtype: int64

=== PROCESSO CONCLUÍDO COM SUCESSO ===


# EXTRAÇÃO DO CICLO DE VIDA (LOTES L E M)

In [4]:
import pandas as pd
df = pd.read_csv('FRUTAS_DB_limpo-corrigido.csv')
print("=== EXTRAINDO CICLO DE VIDA DOS LOTES ===\n")

df = df[
    df['lote'].isin(['lote_L', 'lote_M'])
].copy()

print(" Lotes selecionados:")
print(df['lote'].value_counts())

print("\n Distribuição dos estados:")
print(df['estado_real'].value_counts())

print(f"\n Total de registros: {len(df)}")

clean_file = 'Ciclo-de-vida_FRUTAS_DB.csv'
df.to_csv(clean_file, index=False)
print("\n=== EXPORTAÇÃO FINAL ===\n")
print(f" Arquivo salvo: {clean_file}")
print("\n=== PROCESSO CONCLUÍDO COM SUCESSO ===")

=== EXTRAINDO CICLO DE VIDA DOS LOTES ===

✅ Lotes selecionados:
lote
lote_M    3190
lote_L    1107
Name: count, dtype: int64

✅ Distribuição dos estados:
estado_real
Risco de Perda    1569
Madura            1258
Sem Risco         1115
Alerta             355
Name: count, dtype: int64

✅ Total de registros: 4297

=== EXPORTAÇÃO FINAL ===

✅ Arquivo salvo: Ciclo-de-vida_FRUTAS_DB.csv

=== PROCESSO CONCLUÍDO COM SUCESSO ===


In [5]:
import pandas as pd

df = pd.read_csv('Ciclo-de-vida_FRUTAS_DB.csv')

print("=== COLUNAS DISPONÍVEIS NO DATASET ===\n")
print(df.columns.tolist())

print("\n=== AMOSTRAS DA COLUNA 'validade' ===\n")

display(
    df[
        [
            '_id',
            'lote',
            'estado_real',
            'mq3_raw',
            'temperatura',
            'umidade_ar',
            'validade'
        ]
    ].head(20)
)

print("\n=== ESTATÍSTICAS DA COLUNA 'validade' ===\n")
print(df['validade'].describe())
print("\n=== VALORES ÚNICOS DE 'validade' ===\n")
print(sorted(df['validade'].dropna().unique())[:50])

print(f"\n Total de registros com validade preenchida: {df['validade'].notnull().sum()}")
print(f" Total de registros sem validade: {df['validade'].isnull().sum()}")

=== COLUNAS DISPONÍVEIS NO DATASET ===

['_id', 'tipoFruta', 'temperatura', 'umidade_ar', 'mq3_raw', 'mq3_tensao', 'lote', 'estado_real', 'estado_previsto', 'validade', 'dataRegistro', 'estado_real_original']

=== AMOSTRAS DA COLUNA 'validade' ===



,_id,lote,estado_real,mq3_raw,temperatura,umidade_ar,validade
0,69c2ea664f3e7eb1d6eb0fd4,lote_L,Sem Risco,377,28.5,77,NaN
1,69c2eb924f3e7eb1d6eb0fd5,lote_L,Sem Risco,407,28.5,79,NaN
2,69c2ecbe4f3e7eb1d6eb0fd6,lote_L,Sem Risco,423,28.1,84,NaN
3,69c2edea4f3e7eb1d6eb0fd7,lote_L,Sem Risco,431,28.0,88,NaN
4,69c2ef174f3e7eb1d6eb0fd8,lote_L,Sem Risco,439,28.0,89,NaN
5,69c2f0424f3e7eb1d6eb0fd9,lote_L,Sem Risco,450,28.0,91,NaN
6,69c2f16e4f3e7eb1d6eb0fda,lote_L,Sem Risco,451,28.2,91,NaN
7,69c2f29a4f3e7eb1d6eb0fdb,lote_L,Sem Risco,454,28.5,91,NaN
8,69c2f3c64f3e7eb1d6eb0fdc,lote_L,Sem Risco,454,28.5,92,NaN
9,69c2f4f24f3e7eb1d6eb0fdd,lote_L,Sem Risco,432,28.5,92,NaN



=== ESTATÍSTICAS DA COLUNA 'validade' ===

count    3242.000000
mean       13.269587
std        16.867108
min         0.000000
25%         0.000000
50%         0.000000
75%        30.000000
max        42.000000
Name: validade, dtype: float64

=== VALORES ÚNICOS DE 'validade' ===

[np.float64(0.0), np.float64(12.0), np.float64(30.0), np.float64(42.0)]

✅ Total de registros com validade preenchida: 3242
✅ Total de registros sem validade: 1055


In [6]:
df = pd.read_csv('Ciclo-de-vida_FRUTAS_DB.csv')

df = df.drop(columns=['estado_real_original'])

df = df.rename(columns={
    '_id': 'id_mongo',
    'tipoFruta': 'tipo_fruta',
    'dataRegistro': 'data_registro'
})

df.to_csv(
    'Ciclo-de-vida_FRUTAS_DB_postgres.csv',
    index=False
)

print(" CSV otimizado para PostgreSQL gerado com sucesso.")

✅ CSV otimizado para PostgreSQL gerado com sucesso.


In [7]:
import pandas as pd

df = pd.read_csv('Ciclo-de-vida_FRUTAS_DB_postgres.csv')

mapeamento = {
    'sem risco': 'Sem Risco',
    'alerta': 'Alerta',
    'madura': 'Madura',
    'risco de perda': 'Risco de Perda',

    'sem risco (verde/estável)': 'Sem Risco',
    'alerta (calor excessivo)': 'Alerta',
    'alerta (venda urgente)': 'Alerta',
    'risco de perda (deterioração/mofo)': 'Risco de Perda',
    'risco de perda (sobre-madura)': 'Risco de Perda'
}

df['estado_previsto'] = (
    df['estado_previsto']
    .astype(str)
    .str.lower()
    .str.strip()
    .map(mapeamento)
)

# Exportação final
df.to_csv(
    'Ciclo-de-vida_FRUTAS_DB_FINAL.csv',
    index=False
)

print(" estado_previsto padronizado com sucesso.")

✅ estado_previsto padronizado com sucesso.
